In [ ]:
import tensorflow as tf
import os
import numpy as np
# from osgeo import gdal, osr
# import cv2
import matplotlib.pyplot as plt
import torch.nn.functional as F


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch
import torch.nn as nn
from transformers import SegformerForSemanticSegmentation

class SegFormerSegmentor(nn.Module):
    def __init__(self, num_classes, num_channels=4):
        super(SegFormerSegmentor, self).__init__()
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0",
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        # Adjust input conv layer for different channel numbers (default is 3 for RGB)
        if num_channels != 3:
            old_conv = self.model.segformer.encoder.patch_embeddings[0].proj
            new_conv = nn.Conv2d(num_channels,
                                 old_conv.out_channels,
                                 kernel_size=old_conv.kernel_size,
                                 stride=old_conv.stride,
                                 padding=old_conv.padding,
                                 bias=old_conv.bias is not None)
            # Copy weights for first 3 channels if available, random init for the rest
            with torch.no_grad():
                new_conv.weight[:, :3, :, :] = old_conv.weight
                if num_channels > 3:
                    nn.init.xavier_uniform_(new_conv.weight[:, 3:, :, :])
            self.model.segformer.encoder.patch_embeddings[0].proj = new_conv

    def forward(self, x):
        """
        x: torch.Tensor with shape [B, C, H, W]
        """
        outputs = self.model(x)
        logits = outputs.logits  # [B, num_classes, H/4, W/4] (usually smaller resolution)
        # Upsample back to input resolution
        logits = nn.functional.interpolate(
            logits,
            size=(x.shape[2], x.shape[3]),
            mode="bilinear",
            align_corners=False
        )
        return logits


# Example usage
if __name__ == "__main__":
    model = SegFormerSegmentor(num_classes=10, num_channels=4)
    dummy_input = torch.randn(2, 4, 256, 256)  # [batch, channels, height, width]
    output = model(dummy_input)
    print(output.shape)  # torch.Size([2, 10, 256, 256])

/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch

torch.Size([2, 10, 256, 256])


In [4]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size =1000 #Vermontlc
num_sample = 2000
num_training = 1600
class_num=2
num_test = num_sample-num_training
dataset_name = 'RoadDetections'

In [5]:
file_folder = '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()
output_tfrecords_files

['/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_ATD_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_BiDiff_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_CAMixer_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_CFAT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_ESRGAN_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_RGT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_SED_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_SRNO_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_UPSR_x16.tfrecords']

In [6]:
file_folder = '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()

for output_tfrecords in output_tfrecords_files:
    dirname = '/glade/derecho/scratch/lizhili/s2naip/unet_pths_16x/'  # one level up (/content/drive/MyDrive/GeoSR)
    filename = os.path.basename(output_tfrecords)                 # e.g. M2L8_River_SRNO_x4.tfrecords
    base = filename.replace(f"{dataset_name}_", "").replace("_x16.tfrecords", "")
    segformer_save_path =  os.path.join(dirname, f"{dataset_name}_Segformer_{base}_run2.pth")
    print(output_tfrecords, " Save to:", segformer_save_path)

/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_ATD_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/unet_pths_16x/RoadDetections_Segformer_ATD_run2.pth
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_BiDiff_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/unet_pths_16x/RoadDetections_Segformer_BiDiff_run2.pth
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_CAMixer_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/unet_pths_16x/RoadDetections_Segformer_CAMixer_run2.pth
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_CFAT_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/unet_pths_16x/RoadDetections_Segformer_CFAT_run2.pth
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/RoadDetections_ESRGAN_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/unet_pths_16x/RoadDetections_Segformer_ESRGAN_run2.pth
/gl

In [7]:
def input_pipeline_downstream(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):

    feature_description = {
        'hres': tf.io.FixedLenFeature([4*hres_size_4x*hres_size_4x], dtype=tf.int64),
        'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
    }
    
    @tf.function
    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)

        hres = feature_dict['hres']
        hres = tf.reshape(hres, [4, hres_size_4x, hres_size_4x])
        hres = tf.cast(hres, tf.float32)
        hres = hres/255

        label = feature_dict['label']
        label = tf.reshape(label, [label_size, label_size, 1])

        return hres, label[..., 0]
        
    @tf.function
    def _augment_function(hres_img, label):
        # Transpose to [H, W, C]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
        if tf.rank(label) == 2:
            label = tf.expand_dims(label, axis=-1)
    
        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
        hres_img = tf.image.rot90(hres_img, k=k)
        label = tf.image.rot90(label, k=k)

        # ---- Random horizontal flip ----
        do_flip_lr = tf.random.uniform([]) > 0.5
        hres_img = tf.cond(do_flip_lr,
                           lambda: tf.image.flip_left_right(hres_img),
                           lambda: hres_img)
        label = tf.cond(do_flip_lr,
                        lambda: tf.image.flip_left_right(label),
                        lambda: label)
    
        # ---- Random vertical flip ----
        do_flip_ud = tf.random.uniform([]) > 0.5
        hres_img = tf.cond(do_flip_ud,
                           lambda: tf.image.flip_up_down(hres_img),
                           lambda: hres_img)
        label = tf.cond(do_flip_ud,
                        lambda: tf.image.flip_up_down(label),
                        lambda: label)
    
        # Transpose back to [C, H, W]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
        label = tf.squeeze(label, axis=-1)
    
        return hres_img, label

    dataset = tf.data.TFRecordDataset(filename)
    dataset = dataset.skip(skip)
    if take:
        dataset = dataset.take(take)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)

    return batch

In [8]:
import torch

def random_crop_image_label(
    image,
    label,
    crop_size
):
    """
    Random aligned crop for image–label pairs with strict size checking.

    Args:
        image: Tensor [B, C, H, W]
        label: Tensor [B, H, W] or [B, 1, H, W]
        crop_size: int

    Returns:
        image_crop: [B, C, crop_size, crop_size]
        label_crop: [B, crop_size, crop_size]
    """
    # ---- Shape checks ----
    assert image.dim() == 4, f"image must be [B, C, H, W], got {image.shape}"
    assert label.dim() in (3, 4), f"label must be [B, H, W] or [B, 1, H, W], got {label.shape}"

    _, _, H_img, W_img = image.shape

    if label.dim() == 3:
        _, H_lbl, W_lbl = label.shape
    else:
        _, _, H_lbl, W_lbl = label.shape

    # ---- Enforce same spatial size ----
    assert H_img == H_lbl and W_img == W_lbl, (
        f"Image and label spatial sizes must match, "
        f"got image ({H_img}, {W_img}) and label ({H_lbl}, {W_lbl})"
    )

    assert H_img >= crop_size and W_img >= crop_size, (
        f"Crop size {crop_size} exceeds image size ({H_img}, {W_img})"
    )

    # ---- Random crop ----
    top = torch.randint(0, H_img - crop_size + 1, (1,)).item()
    left = torch.randint(0, W_img - crop_size + 1, (1,)).item()

    image_crop = image[:, :, top:top + crop_size, left:left + crop_size]

    if label.dim() == 3:
        label = label.unsqueeze(1)

    label_crop = label[:, :, top:top + crop_size, left:left + crop_size]
    label_crop = label_crop.squeeze(1).long()

    return image_crop, label_crop

In [ ]:
file_folder = '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
weight_path = '/glade/derecho/scratch/lizhili/s2naip/RoadDetections_NAIP_Segformer_run1_best.pth'
output_tfrecords_files.sort()

for output_tfrecords in output_tfrecords_files:
    dirname = '/glade/derecho/scratch/lizhili/s2naip/segformer_pths_16x/'  # one level up (/content/drive/MyDrive/GeoSR)
    filename = os.path.basename(output_tfrecords)                 # e.g. M2L8_River_SRNO_x4.tfrecords
    base = filename.replace(f"{dataset_name}_", "").replace("_x16.tfrecords", "")
    segformer_save_path =  os.path.join(dirname, f"{dataset_name}_SegFormer_{base}_run1.pth")
    print(output_tfrecords, " Save to:", segformer_save_path)

    print('Begin Segformer Training')
    device = 'cuda'
    model_naip = SegFormerSegmentor(num_classes=class_num, num_channels=4).to(device)
    model_naip.load_state_dict(torch.load(weight_path, weights_only=True))
    print('Weight load successfully')

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model_naip.parameters(), lr=1e-4)

    # Training loop
    def train_step(images, labels):
        model_naip.train()
        # label = tf.image.resize(label, [256, 256], method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)

        images = torch.from_numpy(images.numpy().astype('float32'))
        labels = torch.from_numpy(labels.numpy())

        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(hres_size_4x, hres_size_4x), mode='nearest')
        labels = labels.squeeze(1).to(torch.int64)

        images, labels = random_crop_image_label(images, labels, crop_size = 256)

        images = images.to(device)  # [B, 4, 256, 256]
        labels = labels.to(device)  # [B, 1000, 1000] as class indices

        optimizer.zero_grad()
        logits = model_naip(images)  # [B, 13, 512, 512]
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        return loss.item()

    @torch.no_grad()
    def val_step(images, labels):
        model_naip.eval()
    
        images = torch.from_numpy(images.numpy().astype('float32'))
        labels = torch.from_numpy(labels.numpy())
    
        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(hres_size_4x, hres_size_4x), mode='nearest')
        labels = labels.squeeze(1).to(torch.int64)
    
        images = images.to(device)
        labels = labels.to(device)
    
        logits = model_naip(images)
        loss = loss_fn(logits, labels)
    
        return loss.item()

    def train_test_step(train_loader, epoch):
        total_loss = 0
        for hr, label in train_loader:
            loss = train_step(hr, label)
            total_loss += loss
        avg_loss = total_loss / num_training
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

        if (epoch + 1) % 1 == 0:
            torch.save(model_naip.state_dict(), segformer_save_path)

    best_val_loss = float('inf')

    def train_val_epoch(train_loader, val_loader, epoch):
        global best_val_loss
    
        # ---- Training ----
        total_train_loss = 0.0
        for hr, label in train_loader:
            loss = train_step(hr, label)
            total_train_loss += loss
    
        avg_train_loss = total_train_loss / num_train
    
        avg_val_loss = None  # safeguard
    
        # ---- Validation (every 5 epochs) ----
        if (epoch + 1) % 5 == 0:
            total_val_loss = 0.0
            for hr, label in val_loader:
                loss = val_step(hr, label)
                total_val_loss += loss
    
            avg_val_loss = total_val_loss / num_val
    
            # ---- Checkpoint best model ----
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(
                    model_naip.state_dict(),
                    segformer_save_path.replace(".pth", "_best.pth")
                )
                print("✓ Saved new best model")
    
        # ---- Logging ----
        if (epoch + 1) % 10 == 0:
            log_msg = f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f}"
            if avg_val_loss is not None:
                log_msg += f" | Val Loss: {avg_val_loss:.4f}"
            print(log_msg)

    num_val = int(0.1 * num_training)   # 10% for validation
    num_train = num_training - num_val

    train_loader = input_pipeline_downstream(output_tfrecords, 8, 0, num_train, is_repeat=False)
    val_loader = input_pipeline_downstream(output_tfrecords, 2, num_train, num_val, is_repeat=False)

    for i in range(100):
        # train_test_step(train_loader, i)
        train_val_epoch(train_loader, val_loader, i)


